In [9]:
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

from nilearn import plotting, image, masking


def display_brain_maps(
    image_roi,
    file_name,
    disp_mode,
    slices,
    path2template="/application/fsl/data/standard/",
    thr=98,
    percentile=True,
    smooth=True,
    fwhm=3,
):
    template_file = os.path.join(path2template, "MNI152_T1_2mm.nii.gz")
    template = nib.load(template_file)

    if image_roi.shape != template.shape:
        raise ValueError(
            f"Shape diversa: saliency {image_roi.shape}, template {template.shape}"
        )

    if percentile:
        thr = np.percentile(image_roi, thr)

    binary_mask = masking.compute_brain_mask(
        template_file,
        threshold=0.0
    ).get_fdata()

    image_roi_only_brain = np.zeros_like(image_roi)
    image_roi_only_brain[binary_mask == 1] = image_roi[binary_mask == 1]

    image_nii = nib.Nifti1Image(
        image_roi_only_brain,
        affine=template.affine
    )

    if smooth:
        image_nii = image.smooth_img(image_nii, fwhm=fwhm)

    plotting.plot_stat_map(
        stat_map_img=image_nii,
        bg_img=template,
        threshold=thr,
        cmap=plt.cm.viridis,
        symmetric_cbar=False,
        display_mode=disp_mode,
        cut_coords=slices,
        draw_cross=False,
        output_file=file_name
    )


final_dir = "saliency_maps/MD/final_mean_maps"

ad_map = np.load(os.path.join(final_dir, "MD_final_mean_saliency_AD.npy"))
healthy_map = np.load(os.path.join(final_dir, "MD_final_mean_saliency_Healthy.npy"))

coords = [-15, -6, 0, 6, 15]

for view in ["x", "y", "z"]:
    display_brain_maps(
        ad_map,
        file_name=os.path.join(final_dir, f"AD_{view}_5slices.pdf"),
        disp_mode=view,
        slices=coords
    )

    display_brain_maps(
        healthy_map,
        file_name=os.path.join(final_dir, f"Healthy_{view}_5slices.pdf"),
        disp_mode=view,
        slices=coords
    )

print("Fatto!")

Fatto!
